# Psychoacoustics and Audio Coding
This notebook covers the projects in this folder's README:
1. **Hearing and masking**: the threshold of hearing, critical bands, and how a loud sound hides quieter ones nearby (sections 1 and 2).
2. **Companding**: mu-law, the 8-bit coding of telephone networks (section 3).
3. **Lossless compression**: prediction plus Rice coding, the method FLAC uses (section 4).
4. **Perceptual coding**: a small transform coder that uses a masking model to decide where quantization noise can go, compared with a coder that ignores hearing (section 5).

Encoding to MP3, AAC or FLAC with real codecs needs external programs. The coders here are deliberately small, but use the same ideas.

**Use headphones at a moderate volume** for the listening examples, and start quietly: some of them are loud by design.

In [ ]:
import zlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import (drum_loop, mu_law_compress, mu_law_expand, pluck, sine,
                      synthetic_vowels, time_axis)

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 44100
rng = np.random.default_rng(0)

def play(x, label, fs=FS):
    print(f'{label}:')
    display.display(display.Audio(np.clip(x, -1, 1), rate=fs, normalize=False))

# Test material at 44.1 kHz, peak 0.5 (-6 dBFS)
melody = np.zeros(4 * FS)
for i, f in enumerate([220, 277.2, 329.6, 440, 554.4, 440, 329.6, 277.2, 220, 329.6, 440, 659.3, 880, 659.3, 440, 220]):
    start = int(i * 0.25 * FS)
    melody[start:start + int(0.25 * FS)] += pluck(f, 0.25, FS)
melody *= 0.5 / np.max(np.abs(melody))
drums = 0.5 * drum_loop(110, 2, FS, rng)[0][:4 * FS] / 0.8
vowels_16k, _ = synthetic_vowels('aiueoa', sample_rate=16000)
vowels = signal.resample_poly(vowels_16k, 441, 160)
SIGNALS = {'melody': melody, 'drums': drums, 'vowels': vowels}

## 1. Hearing: threshold and critical bands
**Threshold in quiet.** The quietest tone a young listener can hear depends strongly on frequency. Hearing is most sensitive around 3 to 4 kHz, where the ear canal resonates, and much less sensitive at low and very high frequencies. A common formula for the threshold (Terhardt, 1979), in dB SPL (sound pressure level, 0 dB is about the quietest audible 1 kHz tone), is

$$T_q(f) = 3.64\left(\tfrac{f}{1000}\right)^{-0.8} - 6.5\, e^{-0.6\,(f/1000 - 3.3)^2} + 10^{-3}\left(\tfrac{f}{1000}\right)^4$$

**A-weighting**, used by sound level meters, is a filter that roughly follows the ear's sensitivity at moderate levels.

**Critical bands.** The inner ear analyzes sound with a bank of overlapping band-pass filters. Their bandwidth, the **critical band**, is about 100 Hz below 500 Hz and grows to about 20% of the center frequency above that. Numbering 24 adjacent critical bands gives the **Bark** scale. Sounds within one critical band interact strongly: they mask each other, and their loudness does not simply add. Zwicker's formula converts frequency to Bark:

$$z(f) = 13 \arctan(0.00076 f) + 3.5 \arctan\!\left(\left(\tfrac{f}{7500}\right)^2\right)$$

In [ ]:
def threshold_in_quiet(f):
    """Threshold of hearing in dB SPL (Terhardt, 1979)."""
    k = f / 1000
    return 3.64 * k**-0.8 - 6.5 * np.exp(-0.6 * (k - 3.3)**2) + 1e-3 * k**4

def a_weighting_db(f):
    """A-weighting gain in dB (IEC 61672)."""
    f2 = f**2
    ra = 12194**2 * f2**2 / ((f2 + 20.6**2) * np.sqrt((f2 + 107.7**2) * (f2 + 737.9**2)) * (f2 + 12194**2))
    return 20 * np.log10(ra) + 2.0

def bark(f):
    return 13 * np.arctan(0.00076 * f) + 3.5 * np.arctan((f / 7500)**2)

# Edges of Zwicker's 24 critical bands, in Hz
BAND_EDGES = np.array([0, 100, 200, 300, 400, 510, 630, 770, 920, 1080, 1270, 1480, 1720, 2000,
                       2320, 2700, 3150, 3700, 4400, 5300, 6400, 7700, 9500, 12000, 15500, 22050])

f = np.geomspace(20, 20000, 1000)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].semilogx(f, threshold_in_quiet(f), label='threshold in quiet (dB SPL)')
axes[0].semilogx(f, -a_weighting_db(f), '--', label='inverted A-weighting (dB)')
axes[0].set_ylim(-20, 80)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_title('Sensitivity of hearing')
axes[0].legend()
axes[1].semilogx(f, bark(f))
for edge in BAND_EDGES[1:-1]:
    axes[1].axvline(edge, color='0.85', linewidth=0.8, zorder=0)
axes[1].set_xlabel('Frequency (Hz)   (grey lines: critical band edges)')
axes[1].set_ylabel('Bark')
axes[1].set_title('The Bark scale')
plt.tight_layout()
plt.show()

The tones below all have exactly the same digital level. Listen to how different their loudness is. Laptop speakers cannot play 50 Hz at all, and the 15 kHz tone may be inaudible depending on your age and equipment.

In [ ]:
tone_freqs = [50, 200, 1000, 3500, 10000, 15000]
sequence = np.concatenate([np.concatenate((0.1 * sine(f, 0.7, FS) * np.minimum(1, np.minimum(time_axis(0.7, FS), time_axis(0.7, FS)[::-1]) / 0.02),
                                           np.zeros(int(0.2 * FS)))) for f in tone_freqs])
print('Order:', ', '.join(f'{f} Hz' for f in tone_freqs))
play(sequence, 'Equal-level tones')

## 2. Masking
A loud sound raises the threshold of hearing for quieter sounds near it in frequency: they are **masked**. The effect spreads further upwards in frequency than downwards, so a loud low sound hides higher sounds more easily than the other way round.

A simple model of the masked threshold, used by early perceptual coders, has two parts:
- A **spreading function** in Bark that describes how far the masker's effect reaches (Schroeder, Atal and Hall, 1979). $\Delta z$ is the distance from the masker in Bark:
  $$SF(\Delta z) = 15.81 + 7.5(\Delta z + 0.474) - 17.5\sqrt{1 + (\Delta z + 0.474)^2}\ \text{dB}$$
- An **offset** below the masker's level: about 5.5 dB when the masker is noise-like, and $14.5 + z$ dB when it is a tone, because tones mask noise less well (Johnston, 1988).

Below, the masker is a band of noise one critical band wide around 1 kHz, and the probe is a tone that switches on and off every 0.3 s. Choose the probe's frequency and level, compare the model's prediction with what you hear, and check where the probe becomes audible. The model is a rough approximation of real hearing, so expect a few dB of disagreement.

In [ ]:
def spreading_db(dz):
    """Spreading function in dB for a distance dz in Bark above the masker (Schroeder, Atal and Hall, 1979)."""
    return 15.81 + 7.5 * (dz + 0.474) - 17.5 * np.sqrt(1 + (dz + 0.474)**2)

masker_noise = signal.sosfilt(signal.butter(6, [920, 1080], btype='bandpass', fs=FS, output='sos'), rng.normal(0, 1, 3 * FS))
masker_noise *= 0.1 / np.sqrt(np.mean(masker_noise**2))           # -20 dB re full scale, as RMS
gate = (np.floor(time_axis(3.0, FS) / 0.3) % 2 == 0).astype(float)
gate = np.convolve(gate, np.ones(int(0.01 * FS)) / int(0.01 * FS), mode='same')   # 10 ms smooth switching

def masking_demo(probe_freq=2000, probe_db=-30):
    """probe_db is the probe's level relative to the masker."""
    z_masker = bark(1000)
    freqs = np.geomspace(100, 16000, 500)
    threshold = spreading_db(bark(freqs) - z_masker) - 5.5          # dB re masker, noise masker offset
    predicted = spreading_db(bark(probe_freq) - z_masker) - 5.5

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogx(freqs, threshold, label='masked threshold (model)')
    ax.axvspan(920, 1080, color='0.8', label='masker: noise band')
    ax.plot(probe_freq, probe_db, 'ro', markersize=10, label='probe tone')
    ax.set_ylim(-90, 5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB relative to the masker')
    ax.legend(loc='lower left')
    plt.show()
    verdict = 'audible' if probe_db > predicted else 'masked'
    print(f'Model threshold at {probe_freq} Hz: {predicted:.1f} dB re masker. Probe at {probe_db} dB: predicted {verdict}.')

    probe = 0.1 * np.sqrt(2) * 10 ** (probe_db / 20) * sine(probe_freq, 3.0, FS) * gate   # same RMS scale as the masker
    play(masker_noise + probe, 'Masker with pulsing probe')
    play(probe, 'Probe alone')

display.display(interactive(
    masking_demo,
    probe_freq=widgets.IntSlider(value=2000, min=200, max=8000, step=50, description='Probe (Hz):'),
    probe_db=widgets.IntSlider(value=-30, min=-70, max=0, step=1, description='Probe (dB):'),
))

Try a probe at 1000 Hz, inside the masker's band: at -10 dB it is hidden. Then move it to 2000 Hz and 500 Hz at the same level, an octave above and below. The model predicts a much lower threshold below the masker than above it: masking spreads upwards.

## 3. Companding: mu-law
Telephone networks carry speech as 8 bits per sample at 8 kHz, 64 kbit/s (the G.711 standard). With 8-bit linear quantization the SQNR is about 50 dB for a full-scale signal, and falls by 1 dB for every dB the signal is quieter. A quiet talker would be buried in quantization noise.

**Companding** (compressing, then expanding) fixes this. Before quantizing, the signal passes through a logarithmic curve that boosts small values, and the receiver applies the inverse. In effect the quantization steps are small near zero and large near full scale, so the SQNR stays roughly constant over a wide range of levels. North America and Japan use the mu-law curve, with $\mu = 255$:

$$F(x) = \operatorname{sign}(x)\,\frac{\ln(1 + \mu |x|)}{\ln(1 + \mu)}$$

Europe uses the similar A-law. The code uses the continuous curve. G.711 itself uses a segmented approximation of it.

In [ ]:
def quantize(x, bits):
    """Uniform mid-tread quantizer over [-1, 1)."""
    step = 2 / 2**bits
    return np.clip(np.round(x / step) * step, -1, 1 - step)

def mu_law_8bit(x):
    return mu_law_expand(quantize(mu_law_compress(x), 8))

def sqnr_db(x, y):
    return 10 * np.log10(np.mean(x**2) / np.mean((y - x)**2))

levels_db = np.arange(-60, 1, 2)
test = sine(997, 1.0, 8000)
linear = [sqnr_db(10**(L / 20) * test, quantize(10**(L / 20) * test, 8)) for L in levels_db]
mu = [sqnr_db(10**(L / 20) * test, mu_law_8bit(10**(L / 20) * test)) for L in levels_db]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(levels_db, linear, label='8-bit linear')
ax.plot(levels_db, mu, label='8-bit mu-law')
ax.set_xlabel('Sine level (dB re full scale)')
ax.set_ylabel('SQNR (dB)')
ax.set_title('Quantization noise vs signal level')
ax.legend()
plt.show()
for L in (0, -20, -40):
    i = list(levels_db).index(L)
    print(f'At {L:>3} dBFS: linear {linear[i]:5.1f} dB, mu-law {mu[i]:5.1f} dB')

quiet_talker = 10 ** (-30 / 20) * signal.resample_poly(vowels_16k, 1, 2) / 0.5   # 8 kHz, peak at -30 dBFS
play(quantize(quiet_talker, 8) / 10 ** (-30 / 20) * 0.5, '8-bit linear, quiet talker (amplified for listening)', fs=8000)
play(mu_law_8bit(quiet_talker) / 10 ** (-30 / 20) * 0.5, '8-bit mu-law, quiet talker (amplified for listening)', fs=8000)

## 4. Lossless compression: prediction plus Rice coding
A lossless coder must reproduce every bit, so it cannot remove anything. It can only describe the samples more efficiently. FLAC does it in two steps:
1. **Prediction.** Neighboring audio samples are similar, so each sample is predicted from the previous ones and only the prediction error, the **residual**, is stored. FLAC's simplest "fixed" predictors are polynomial: order 0 stores the sample itself, order 1 predicts $x[n-1]$, order 2 predicts $2x[n-1] - x[n-2]$ (a straight line), order 3 a parabola. Their residuals are the first, second and third differences of the signal. The decoder undoes the prediction exactly, because it uses only integer arithmetic.
2. **Rice coding.** Residuals are mostly small, so they should get short codes. A Rice code with parameter $k$ writes the value's low $k$ bits directly and the rest of the value in unary (a run of ones ending in a zero). Small values take about $k + 1$ bits. Negative values are first interleaved with positive ones (0, -1, 1, -2, 2, ... become 0, 1, 2, 3, 4, ...).

For each block of 4096 samples, the coder below tries every predictor order and every $k$, and keeps the combination that needs the fewest bits. The result is compared with `zlib` (the general-purpose compressor in zip files) on the same bytes.

In [ ]:
def to_int16(x):
    """Quantize to 16 bits with TPDF dither, as a recording would be."""
    d = (rng.uniform(-0.5, 0.5, len(x)) + rng.uniform(-0.5, 0.5, len(x)))
    return np.clip(np.round(x * 32767 + d), -32768, 32767).astype(np.int64)

def zigzag(e):
    return np.where(e >= 0, 2 * e, -2 * e - 1)

def rice_bits(e, k):
    return int(np.sum((zigzag(e) >> k) + 1 + k))

def encode_block_bits(block):
    """Bits for the best (order, k) choice: order-many raw 16-bit warm-up samples plus the Rice-coded residual."""
    best = None
    for order in range(4):
        residual = np.diff(block, n=order) if order else block
        for k in range(0, 17):
            bits = 2 + 5 + 16 * order + rice_bits(residual, k)       # header: order (2 bits), k (5 bits)
            if best is None or bits < best[0]:
                best = (bits, order, k)
    return best

def undo_prediction(warmup, residual, order):
    """Rebuild a block from its first `order` samples and its order-th difference."""
    x = residual
    for level in range(order, 0, -1):
        start = np.diff(warmup, n=level - 1)[0] if level > 1 else warmup[0]
        x = np.concatenate(([start], start + np.cumsum(x)))
    return x

def lossless_report(name, samples, block_size=4096):
    total_bits, orders, exact = 0, [], True
    for start in range(0, len(samples), block_size):
        block = samples[start:start + block_size]
        bits, order, k = encode_block_bits(block)
        total_bits += bits
        orders.append(order)
        residual = np.diff(block, n=order) if order else block
        exact &= np.array_equal(undo_prediction(block[:order], residual, order) if order else residual, block)
    raw_bits = 16 * len(samples)
    zlib_bits = 8 * len(zlib.compress(samples.astype('<i2').tobytes(), 9))
    print(f'{name:<16}{raw_bits / total_bits:>8.2f}:1{raw_bits / zlib_bits:>9.2f}:1'
          f'{total_bits / len(samples):>12.2f}   {np.bincount(orders, minlength=4)}   {"yes" if exact else "NO"}')

print(f'{"Signal":<16}{"Rice":>10}{"zlib":>11}{"bits/sample":>12}   blocks per order 0-3   exact')
lossless_report('melody', to_int16(melody))
lossless_report('drums', to_int16(drums))
lossless_report('vowels', to_int16(vowels))
lossless_report('white noise', to_int16(rng.uniform(-0.5, 0.5, 4 * FS)))
lossless_report('silence + dither', to_int16(np.zeros(4 * FS)))

"exact" checks that undoing the prediction gives back every sample. The Rice code is reversible by construction. The short check below encodes one block into an actual bit string and decodes it.

The numbers show what lossless coding can and cannot do. Predictable material (the melody, the vowels) compresses well. White noise cannot be predicted, so it barely compresses at all: no lossless coder can shrink it much. Real music usually lands around 1.5:1 to 2:1. A general-purpose compressor like zlib does much worse on audio, because it looks for repeated byte patterns, which audio rarely has.

In [ ]:
def rice_encode(e, k):
    return ''.join('1' * (u >> k) + '0' + format(u & ((1 << k) - 1), f'0{k}b') if k else '1' * u + '0' for u in zigzag(e))

def rice_decode(bits, k, count):
    values, i = [], 0
    for _ in range(count):
        q = 0
        while bits[i] == '1':
            q, i = q + 1, i + 1
        i += 1
        r = int(bits[i:i + k], 2) if k else 0
        i += k
        u = (q << k) | r
        values.append(u // 2 if u % 2 == 0 else -(u + 1) // 2)
    return np.array(values)

block = to_int16(melody)[FS:FS + 4096]
_, order, k = encode_block_bits(block)
residual = np.diff(block, n=order) if order else block
bitstring = rice_encode(residual, k)
decoded = rice_decode(bitstring, k, len(residual))
print(f'Block of 4096 samples: order {order}, k = {k}, {len(bitstring)} bits '
      f'({len(bitstring) / len(residual):.2f} bits per residual). Decoded exactly: {np.array_equal(decoded, residual)}')

## 5. Perceptual coding
A lossy coder may change the signal, as long as the changes cannot be heard. MP3, AAC and Opus all follow the same plan:
1. **Transform** the signal into frequency bands. These codecs use the **MDCT** (modified discrete cosine transform): frames of $2M$ samples overlapping by half, each giving $M$ coefficients. With a suitable window (here a sine window), the overlapping parts cancel each other's errors and the inverse gives back the input exactly (**time-domain aliasing cancellation**), even though there are no more coefficients than samples.
2. **Model hearing.** For each frame, estimate the masked threshold in each critical band from the spreading function and offsets in section 2, and the threshold in quiet from section 1.
3. **Quantize** each band coarsely enough that its quantization noise just stays under the masked threshold. Uniform quantization with step $\Delta$ adds noise of power $\Delta^2/12$ per coefficient, so the step for each band follows from its allowed noise.
4. **Entropy-code** the quantized values. Here the bit count is estimated from the entropy of the quantized values in each band, which a good entropy coder comes close to, plus 6 bits per band per frame for the step sizes.

The threshold in quiet is in dB SPL, so the coder must assume a playback level. Like the MPEG reference model, it assumes that a full-scale sine plays at 96 dB SPL.

The comparison coder spends the **same number of bits** with one step size for all coefficients, which minimizes the total error power but ignores hearing. The **noise-to-mask ratio** (NMR) measures the result in each band of each frame: the quantization noise relative to the masked threshold. Above 0 dB, the model predicts the noise is audible.

In [ ]:
M = 1024                                       # coefficients per frame; frames are 2M samples, hop M
N_FRAME = 2 * M
_n, _k = np.arange(N_FRAME), np.arange(M)
MDCT_WINDOW = np.sin(np.pi * (_n + 0.5) / N_FRAME)
MDCT_BASIS = np.cos(np.pi / M * (_n[None, :] + 0.5 + M / 2) * (_k[:, None] + 0.5))

def mdct(x):
    padded = np.concatenate((np.zeros(M), x, np.zeros(M + (-len(x)) % M)))
    frames = np.lib.stride_tricks.sliding_window_view(padded, N_FRAME)[::M]
    return (frames * MDCT_WINDOW) @ MDCT_BASIS.T

def imdct(X, length):
    frames = (X @ MDCT_BASIS) * MDCT_WINDOW * (2 / M)
    out = np.zeros((len(X) - 1) * M + N_FRAME)
    for i, frame in enumerate(frames):
        out[i * M:i * M + N_FRAME] += frame
    return out[M:M + length]

print(f'MDCT then inverse MDCT, max error: {np.max(np.abs(imdct(mdct(melody), len(melody)) - melody)):.1e}')

# Band layout: which MDCT coefficients fall in each critical band
coef_freqs = (_k + 0.5) * FS / (2 * M)
band_index = np.searchsorted(BAND_EDGES, coef_freqs, side='right') - 1
n_bands = len(BAND_EDGES) - 1
band_size = np.bincount(band_index, minlength=n_bands)
band_bark = bark((BAND_EDGES[:-1] + BAND_EDGES[1:]) / 2)
band_quiet_db = np.array([threshold_in_quiet(np.maximum(coef_freqs[band_index == b], 20)).min() for b in range(n_bands)])

# Calibration: the energy one frame gets from a full-scale sine corresponds to 96 dB SPL
REF_ENERGY = np.median(np.sum(mdct(sine(1000, 1.0, FS))**2, axis=1))

def band_energy(X):
    return np.stack([np.sum(X[:, band_index == b]**2, axis=1) for b in range(n_bands)], axis=1)

def allowed_noise(X):
    """Masked threshold per frame and band, as an energy in MDCT units."""
    spl = 96 + 10 * np.log10(band_energy(X) / REF_ENERGY + 1e-20)
    dz = band_bark[None, :] - band_bark[:, None]                        # [masker, maskee]
    contributions = spl[:, :, None] + spreading_db(dz)[None] - (14.5 + band_bark)[None, :, None]
    masked_db = 10 * np.log10(np.sum(10 ** (contributions / 10), axis=1))
    threshold_db = np.maximum(masked_db, band_quiet_db[None, :] + 10 * np.log10(band_size)[None, :])
    return REF_ENERGY * 10 ** ((threshold_db - 96) / 10)

def estimated_bits(Q):
    """Entropy of the quantized values, band by band, plus 6 bits of step size per band per frame."""
    bits = 6 * n_bands * len(Q)
    for b in range(n_bands):
        values = Q[:, band_index == b].ravel()
        _, counts = np.unique(values, return_counts=True)
        p = counts / counts.sum()
        bits += len(values) * -np.sum(p * np.log2(p))
    return bits

def perceptual_coder(X, quality_factor):
    allowed = allowed_noise(X)
    step = np.sqrt(12 * allowed / band_size[None, :]) * np.sqrt(quality_factor)
    step_per_coef = step[:, band_index]
    Q = np.round(X / step_per_coef)
    return Q * step_per_coef, estimated_bits(Q)

def uniform_coder(X, target_bits):
    lo, hi = 1e-3, 1e4                                  # bisection on the single step size, in log space
    for _ in range(40):
        step = np.sqrt(lo * hi)
        if estimated_bits(np.round(X / step)) > target_bits:
            lo = step
        else:
            hi = step
    step = np.sqrt(lo * hi)
    Q = np.round(X / step)
    return Q * step, estimated_bits(Q)

def nmr_db(X, X_hat):
    noise = band_energy(X_hat - X)
    return 10 * np.log10(noise / allowed_noise(X) + 1e-20)

In [ ]:
def coder_demo(name='melody', quality_factor=4.0):
    x = SIGNALS[name]
    X = mdct(x)
    X_perc, bits_perc = perceptual_coder(X, quality_factor)
    X_unif, bits_unif = uniform_coder(X, bits_perc)
    y_perc, y_unif = imdct(X_perc, len(x)), imdct(X_unif, len(x))
    duration = len(x) / FS

    print(f'Original: {16 * FS / 1000:.0f} kbit/s (16-bit mono at 44.1 kHz)')
    print(f'{"Coder":<12}{"kbit/s":>8}{"SNR":>10}{"bands > 1 dB over threshold":>30}{"mean excess":>13}')
    for label, X_hat, y, bits in [('perceptual', X_perc, y_perc, bits_perc), ('uniform', X_unif, y_unif, bits_unif)]:
        nmr = nmr_db(X, X_hat)
        loud = band_energy(X) > 1e-6 * REF_ENERGY                      # ignore silent bands
        print(f'{label:<12}{bits / duration / 1000:>8.1f}{sqnr_db(x, y):>7.1f} dB'
              f'{100 * np.mean(nmr[loud] > 1):>28.1f} %{np.mean(np.maximum(nmr[loud], 0)):>10.2f} dB')

    frame = np.argmax(np.sum(X**2, axis=1) * (np.arange(len(X)) > 5))   # a loud frame
    spl = lambda E: 96 + 10 * np.log10(E / REF_ENERGY + 1e-20)
    centers = (BAND_EDGES[:-1] + BAND_EDGES[1:]) / 2
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.semilogx(coef_freqs, spl(X[frame]**2), color='0.7', linewidth=0.8, label='signal, per coefficient')
    ax.semilogx(centers, spl(allowed_noise(X)[frame] / band_size), 'k-', linewidth=2, label='masked threshold, per coefficient')
    ax.semilogx(centers, spl(band_energy(X_perc - X)[frame] / band_size), 'o-', label='noise, perceptual coder')
    ax.semilogx(centers, spl(band_energy(X_unif - X)[frame] / band_size), 's-', label='noise, uniform coder')
    ax.set_xlim(50, FS / 2)
    ax.set_ylim(-20, 100)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB SPL (assumed playback level)')
    ax.set_title(f'One frame of "{name}"')
    ax.legend(loc='lower left', fontsize=9)
    plt.show()
    play(x, 'Original')
    play(y_perc, 'Perceptual coder')
    play(y_unif, 'Uniform coder, same bit count')

display.display(interactive(
    coder_demo,
    name=widgets.Dropdown(options=list(SIGNALS), description='Signal:'),
    quality_factor=widgets.FloatLogSlider(value=1.0, base=2, min=-1, max=6, step=0.5, description='Noise x:'),
))

**Noise x** multiplies the allowed noise power. At 1 the coder places its noise at the modeled threshold: the bit rate the model considers transparent. Higher values save bits by letting the noise rise above the threshold. The table counts the bands whose noise is more than 1 dB over the threshold (with noise placed right at the threshold, rounding puts some just above it), and the average excess over the threshold.

What to look for:
- The **uniform coder always has the higher SNR** at the same bit count. That is guaranteed, because one step size for every coefficient minimizes the total error power. SNR is the wrong measure for perceptual coding.
- **Near transparency** (noise factor 1 or below), the perceptual coder keeps its noise under the threshold, while the uniform coder, with the same bits, lets noise over it in some bands. The difference is largest on the drums.
- **At higher noise factors** the simple perceptual coder lets every band exceed the threshold by the same amount, and on the melody and vowels the uniform coder then does better on the excess measure. Real encoders run an iterative rate loop that decides which bands can best afford the extra noise, instead of raising them all equally. Listen for the "underwater" or "metallic" sound of an overloaded coder, as bands switch on and off from frame to frame.
- On the **drums**, listen for **pre-echo**: a 2048-sample frame spreads the quantization noise over 46 ms, so it can be heard just before each sharp hit, where nothing masks it. Real codecs switch to short frames when they detect a transient.
- The model assumes a full-scale sine plays at 96 dB SPL. At that level the threshold in quiet is about 90 dB below full scale between 2 and 5 kHz, so the coder spends many of its bits on faint high-frequency detail that would be audible only at high playback levels.